# 07 Audit log and threat model

**Purpose.** Demonstrate the SHA-256 hash-chain audit log and test which manipulations it detects.

**Threat model in brief.** The chain attests to the order and integrity of records *as written*. Full tamper detection depends on an independently protected **head digest and record count**:
without that anchor, an adversary with write access to the whole log can edit a record and recompute every later link, and internal verification passes. The chain says nothing about the correctness of the verdicts it records, provides integrity but not confidentiality, and gives order but not trusted time.
The mechanism supports tamper-evident decision provenance relevant to healthcare audit requirements, without establishing compliance with any specific regulatory framework.

In [1]:
# --- setup: work from the laptop/ folder so that relative paths resolve ---
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
pd.set_option("display.precision", 4)
print("working folder:", ROOT)

working folder: <repo>\local_analysis


In [2]:
import random
from audit import HashChainLog, make_event
rng = random.Random(0)
chain = HashChainLog()
for i in range(1000):
    chain.append(make_event(i, rng))
head, n = chain.head, len(chain.records)          # the anchor: stored OUTSIDE the log
print("valid chain:", chain.verify(), " head:", head[:16], "... records:", n)

chain.records[500]["payload"]["predicted_class"] ^= 1          # in-place edit
print("after in-place edit:", chain.verify())

valid chain: (True, None)  head: d85abb693c5862fd ... records: 1000
after in-place edit: (False, 500)


In [3]:
# recompute-after-edit: internal verification passes, the external anchor catches it
def rehash(c, start):
    prev = c.records[start - 1]["hash"] if start else HashChainLog.GENESIS
    for r in c.records[start:]:
        r["prev"] = prev; r["hash"] = HashChainLog.digest(prev, r["payload"]); prev = r["hash"]
rehash(chain, 500)
print("internal verification after recomputing the links:", chain.verify())
print("anchored head still matches:", chain.head == head, "-> tampering", "NOT detected" if chain.head == head else "detected")

internal verification after recomputing the links: (True, None)
anchored head still matches: False -> tampering detected


In [4]:
import subprocess
subprocess.run([sys.executable, "audit_threat_model.py"], capture_output=True)
pd.DataFrame(json.load(open("results/audit_threat_model.json"))).T

,detected_internal_only_pct,detected_with_external_anchor_pct,trials
recompute,0,100,100
truncate_continue,0,100,100
recompute_last_only,0,100,100
